In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


np.random.seed(42)

# Using Path().resolve() for Notebook consistency (mimics __file__ for interactive environments)
# Traverse up two levels from training/mcmc to project root



PROJECT_ROOT = Path().resolve().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MOCK_CSV_PATH = DATA_DIR / "mock_drill_logs.csv"

In [ ]:
n_boreholes = 25
data = {
    "borehole_id": [f"BH-{i+1:03d}" for i in range(n_boreholes)],
    "depth_m": np.random.uniform(10, 150, n_boreholes).round(1),
    "rmr_score": np.clip(np.random.normal(52, 12, n_boreholes), 0, 100).round(1),
    "rock_class": np.random.choice(["II", "III", "IV", "V"], n_boreholes, p=[0.1, 0.4, 0.35, 0.15]),
    "ucs_mpa": np.clip(np.random.lognormal(3.8, 0.4, n_boreholes), 5, 200).round(1),
    "rqd_pct": np.clip(np.random.beta(5, 3, n_boreholes) * 100, 0, 100).round(1),
}

# Inject missing data — reality
data["ucs_mpa"][3] = np.nan
data["ucs_mpa"][17] = np.nan
data["rqd_pct"][8] = np.nan

df = pd.DataFrame(data)


df.to_csv(MOCK_CSV_PATH, index=False)


df.head()

In [38]:
class DrillLogPipeline: # lets keep it super specific to our csv
    """Ingests, validates, and formats drill-log data for PyMC models."""

    def __init__(self, filepath: Path ) -> None:
        self.filepath = filepath
        self.raw_data = None
        self.clean_data = None


    def load(self) -> pd.DataFrame:
        """load the raw csv"""
        self.raw_data = pd.read_csv(self.filepath)
        print(f"Loaded {len(self.raw_data)} rows from {self.filepath}")
        return self.raw_data

    def check_null(self):
        """High-signal Forensic Audit: only reports on Data Voids (NaNs)."""
        null_counts = self.raw_data.isnull().sum()
        # Filter for only parameters containing voids
        self.data_voids = null_counts[null_counts > 0]

        if self.data_voids.empty:
            print("Forensic Audit: No data voids detected.")
        else:
            print("Forensic Alert: Data voids found in high-priority parameters:")
            for param, count in self.data_voids.items():
                print(f"  - {param}: {count} null values")
        return self.data_voids

    def check_range(self, column_targets: list | dict, min_val: float = 0, max_val: float = 100, clipto_bounds: bool = False):
        """
        Checks columns against physical ranges.
        Provide a list to use default min/max bounds, or a dict mapping targets to (min, max) tuples.
        """
        results = {}

        # Standardize input to dictionary mapping for uniform processing
        if isinstance(column_targets, list):
            column_targets = {col: (min_val, max_val) for col in column_targets}

        for target, (target_min, target_max) in column_targets.items():
            # Map index to name if integer provided
            col_name = self.raw_data.columns[target] if isinstance(target, int) else target

            # Vectorized Range Audit
            out_of_range = np.sum((self.raw_data[col_name] < target_min) | (self.raw_data[col_name] > target_max)).item()
            results[col_name] = out_of_range

            if out_of_range > 0:
                print(f"Forensic Alert: [{col_name}] has {out_of_range} values outside [{target_min}, {target_max}]")

            if clipto_bounds:

                self.raw_data[col_name] = self.raw_data[col_name].clip(lower=target_min, upper=target_max)
                if out_of_range > 0:
                    print(f"  -> Action Taken: Clipped out-of-bounds values for [{col_name}] to [{target_min}, {target_max}].")

        return results


    def clean(self, drop_na: bool = False, reset_index: bool = True) -> pd.DataFrame:
        """Clean the data for modeling."""
        df = self.raw_data.copy()

        if drop_na:
            print("\n[!] FORENSIC ALERT: Prime Directive Violation. Dropping data voids instead of interrogating them.")
            # Dynamically identify columns containing Data Voids (NaNs)
            na_cols = df.columns[df.isnull().any()].tolist()
            if na_cols:
                df = df.dropna(subset=na_cols)
                print(f"Dropped rows with missing values in {na_cols}. Remaining: {len(df)}")

        if reset_index:
            df = df.reset_index(drop=True)

        self.clean_data = df
        return df


    def to_pymc(self, column: str) -> np.ndarray:
        """Extract a clean NumPy array ready for PyMC observed= parameter."""
        if self.clean_data is None:
            raise ValueError("Run .clean() first!")
        return self.clean_data[column].values


    def get_summary(self) -> pd.DataFrame:
        """High-signal statistical readout for pre-MCMC variance checks."""
        if self.clean_data is None:
            raise ValueError("Run .clean() first!")

        print("\n--- Operational Data Summary ---")
        # Transpose it for vertical readability and focus heavily on risk metrics (count, mean, std, boundaries)
        summary = self.clean_data.describe().T[['count', 'mean', 'std', 'min', 'max']]
        return summary.round(1)

    def suggest_priors(self, column: str) -> dict:
        """
        Suggest weakly informative priors based on the empirical data range.
        Calculates and formats both Normal and Uniform distributions so you can choose based on physics.
        """
        # 1. Fetch exactly the vector PyMC will see (avoids NaN bugs)
        data = self.to_pymc(column)

        # 2. Extract empirical forensics
        emp_mean = np.mean(data)
        emp_std = np.std(data)
        emp_min = np.min(data)
        emp_max = np.max(data)

        # 3. Parameter logic: Weakly informative
        n_mu = round(float(emp_mean), 2)
        n_sig = round(float(emp_std * 2.0), 2) # Double std dev for Normal
        u_low = round(float(emp_min * 0.8), 2) # -20% buffer for Uniform
        u_high = round(float(emp_max * 1.2), 2) # +20% buffer for Uniform

        # Output a high-signal text readout for the user to copy-paste into PyMC
        print(f"\n[+] Prior Orchestration for '{column}':")
        print(f"  -> Empirical Data : mean={emp_mean:.1f} | std={emp_std:.1f} | range=[{emp_min:.1f}, {emp_max:.1f}]")
        print(f"  [Option A] Normal prior (Use if physics allow symmetric variance without hitting 0):")
        print(f"      pm.Normal('{column}_prior', mu={n_mu}, sigma={n_sig})")
        print(f"  [Option B] Uniform prior (Use if you have hard structural min/max boundaries):")
        print(f"      pm.Uniform('{column}_prior', lower={u_low}, upper={u_high})")

        return {"normal": (n_mu, n_sig), "uniform": (u_low, u_high)}

    def encode_rock_class(self) -> np.ndarray:
        """Convert rock class strings to ordered integers for PyMC."""
        mapping = {"I": 0, "II": 1, "III": 2, "IV": 3, "V": 4}
        return self.clean_data["rock_class"].map(mapping).values

    def __repr__(self):
        n = len(self.raw_data) if self.raw_data is not None else 0
        return f"<DrillLogPipeline('{self.filepath}', n={n})>"

    # since the obj holds complex loaded states, will make it a diagnostic readout rather than conventional python

In [ ]:

# Verifying the Pipeline


print("INITIALIZATION")
pipeline = DrillLogPipeline(MOCK_CSV_PATH)
pipeline.load()

print("\nAUDIT")
# Check for NaNs
pipeline.check_null()
# Check ranges (cols 2=rmr_score, 4=ucs_mpa, 5=rqd_pct)
pipeline.check_range([2, 4, 5])

print("\nCLEANING")
# keeping drop_na=True here to see the warning and action)
pipeline.clean(drop_na=True)

print("\nSUMMARY")
display(pipeline.get_summary())

print("\nCATEGORICAL ENCODING")
# Test our rock class encoder
rock_idx = pipeline.encode_rock_class()
print(f"Mapped Rock Classes for PyMC (first 10): {rock_idx[:10]}")

print("\nBAYESIAN PRIOR SUGGESTIONS")
# Test our new weakly informative prior generator for UCS and RMR
ucs_priors = pipeline.suggest_priors("ucs_mpa")
rmr_priors = pipeline.suggest_priors("rmr_score")

print("\nVECTOR EXTRACTION")
# extract the ready 1D numpy arrays
ucs_data = pipeline.to_pymc("ucs_mpa")
rmr_data = pipeline.to_pymc("rmr_score")
print(f"UCS array shape: {ucs_data.shape}, dtype: {ucs_data.dtype}")
print(f"RMR array shape: {rmr_data.shape}, dtype: {rmr_data.dtype}")